In [1]:
import os
import sys
from datetime import date, datetime
from dotenv import load_dotenv
# Configuración de rutas
sys.path.append("../src")
# from repositories import *
from models import *



load_dotenv()

from domain.config import DB_URL
from repositories import UnitOfWorkFactory
from models import (
    Region, Player, Team, VideoGameType, 
    TournamentMode, TournamentStatus, Tournament, Match
)
from repositories import (
    PlayerRepository, TeamRepository, TournamentRepository, 
    RegionRepository, VideoGameTypeRepository
)

print(f"✅ Entorno listo. Conectado a: {DB_URL}")

# Inicialización de la fábrica
uow_factory = UnitOfWorkFactory(DB_URL)

# La función de ayuda ahora llama a .create()
def get_uow():
    return uow_factory.create()

✅ Entorno listo. Conectado a: sqlite:///./test_db.sqlite


In [2]:
# --- FORZAR CREACIÓN DE TABLAS (Añadir al final de la Celda 1) ---
from domain.db import Base

# Usamos el engine de la factoría para crear lo que falte
Base.metadata.create_all(bind=uow_factory.engine)

print("✅ Tablas verificadas/creadas en la base de datos.")

✅ Tablas verificadas/creadas en la base de datos.


PlayerTeam + TeamTournament


In [3]:
print("--- TESTING Intermediate Repositories (Many-to-Many) ---")

with get_uow() as uow:
    from repositories import PlayerTeamRepository, TeamTournamentRepository
    from models import PlayerTeam, TeamTournament
    from datetime import date
    
    pt_repo = PlayerTeamRepository(uow, PlayerTeam)
    tt_repo = TeamTournamentRepository(uow, TeamTournament)
    
    # --- PASO 0: LIMPIEZA (Para evitar el UNIQUE constraint error) ---
    # Borramos registros previos de estas tablas para que el test empiece de cero
    uow.session.query(PlayerTeam).delete()
    uow.session.query(TeamTournament).delete()
    uow.commit() 
    print("🧹 Tablas intermedias limpiadas para el test.")

    # --- 1. TEST PLAYER-TEAM ---
    # CREACIÓN (Ahora ya no fallará porque acabamos de borrar el anterior)
    vinc_pt = PlayerTeam(id_player=1, id_team=1, registered_at=date.today())
    pt_repo.add(vinc_pt)
    uow.commit()
    print("[OK] PlayerTeam: Registro creado.")

    # LECTURA ALL
    todos_pt = pt_repo.get_all()
    print(f"[OK] PlayerTeam: Leídos {len(todos_pt)} registros.")

    # --- 2. TEST TEAM-TOURNAMENT ---
    vinc_tt = TeamTournament(id_team=1, id_tournament=1, points_obtained=500)
    tt_repo.add(vinc_tt)
    uow.commit()
    print("[OK] TeamTournament: Registro creado.")

    # BORRADO (Opcional: puedes borrarlo al final para dejar la DB limpia)
    # pt_repo.delete(vinc_pt)
    # uow.commit()
    # print("[OK] Limpieza final realizada.")

print("\n¡Test completado con éxito y sin errores de duplicados!")

--- TESTING Intermediate Repositories (Many-to-Many) ---
🧹 Tablas intermedias limpiadas para el test.
[OK] PlayerTeam: Registro creado.
[OK] PlayerTeam: Leídos 1 registros.
[OK] TeamTournament: Registro creado.

¡Test completado con éxito y sin errores de duplicados!


PlayerProfile + Match

In [4]:
print("--- TESTING Profile & Match ---")
with get_uow() as uow:
    from repositories import PlayerProfileRepository, MatchRepository
    profile_repo = PlayerProfileRepository(uow.session, PlayerProfile)
    match_repo = MatchRepository(uow.session, Match)
    
    # 1. Player Profile (Creation)
    perfil = PlayerProfile(id_player=1, bio="Pro Player de CS2", avatar_url="http://link.com")
    profile_repo.add(perfil)
    
    # 2. Match (Creation)
    # Creamos un partido entre dos equipos (Team 1 vs Team 2)
    partido = Match(
        id_tournament=1,
        id_team_home=1,
        id_team_away=2, # Asegúrate de que exista un segundo equipo
        match_date=datetime.now(),
        score_home=16,
        score_away=12
    )
    match_repo.add(partido)
    
    # 3. Match Specific Query: Listar partidos de un torneo
    partidos_torneo = match_repo.get_matches_by_tournament(tournament_id=1)
    
    uow.commit()
    print(f"[OK] Profile & Match: Creados y {len(partidos_torneo)} partidos filtrados")

--- TESTING Profile & Match ---


ImportError: cannot import name 'PlayerProfileRepository' from 'repositories' (c:\Users\valde\Desktop\federico\notebooks\../src\repositories\__init__.py)

TournamentMode + TournamentStatus

In [ ]:
print("--- TESTING Catalog Repositories ---")
with get_uow() as uow:
    from repositories import TournamentModeRepository, TournamentStatusRepository
    mode_repo = TournamentModeRepository(uow.session, TournamentMode)
    status_repo = TournamentStatusRepository(uow.session, TournamentStatus)
    
    # Reading all (para demostrar que los catálogos funcionan)
    modos = mode_repo.get_all()
    estados = status_repo.get_all()
    
    print(f"[OK] Catalogs: {len(modos)} modos y {len(estados)} estados leídos.")

Region

In [ ]:
print("--- TESTING RegionRepository ---")
with get_uow() as uow:
    repo = RegionRepository(uow.session, Region)
    
    # 1. Creation
    nueva_reg = Region(name="Latam North", country_code="LN")
    repo.add(nueva_reg)
    uow.commit()
    reg_id = nueva_reg.id_region
    print(f"[OK] Creation: ID {reg_id}")

    # 2. Reading by ID
    reg = repo.get(reg_id)
    print(f"[OK] Read by ID: {reg.name}")

    # 3. Reading all
    todas = repo.get_all()
    print(f"[OK] Read all: {len(todas)} regiones encontradas")

    # 4. Updating
    reg.name = "Latam North Updated"
    uow.commit()
    print(f"[OK] Updating: {repo.get(reg_id).name}")

    # 5. Specific Query (count_players)
    count = repo.count_players(reg_id)
    print(f"[OK] Specific Query (count_players): {count} jugadores")
    
    # 6. Deleting (opcional probarlo aquí o al final)
    # repo.delete(reg)
    # uow.commit()

--- TESTING RegionRepository ---


AttributeError: 'Session' object has no attribute 'session'

Player

In [ ]:
print("--- TESTING PlayerRepository ---")
with get_uow() as uow:
    player_repo = PlayerRepository(uow.session, Player)
    
    # 1. Creation
    p1 = Player(nickname="Gamer1", email="g1@test.com", birth_date=date(1990,1,1), 
                registration_date=datetime.now(), id_region=1)
    p2 = Player(nickname="Gamer2", email="g2@test.com", birth_date=date(1992,2,2), 
                registration_date=datetime.now(), id_region=1)
    player_repo.add(p1)
    player_repo.add(p2)
    uow.commit()
    print(f"[OK] Creation: Jugadores creados")

    # 2. Pagination (Requisito)
    page = player_repo.get_paginated(page=1, page_size=1)
    print(f"[OK] Pagination: {len(page)} registro(s) en pág 1")

    # 3. Concrete Domain Operation (Requisito)
    # Supongamos que ya existe un equipo con ID 1
    try:
        player_repo.add_player_to_team_direct(player_id=p1.id_player, team_id=1)
        uow.commit()
        print(f"[OK] Domain Op: Jugador vinculado a equipo")
    except Exception as e:
        print(f"[INFO] Domain Op: Saltado (necesita Team ID 1)")

    # 4. Deleting
    player_repo.delete(p2)
    uow.commit()
    print(f"[OK] Deleting: Gamer2 eliminado")

Tournament + VideoGameType

In [ ]:
print("--- TESTING Tournament & VideoGameType ---")
with get_uow() as uow:
    vgt_repo = VideoGameTypeRepository(uow.session, VideoGameType)
    tour_repo = TournamentRepository(uow.session, Tournament)
    
    # Crear tipo de juego
    vgt = VideoGameType(name="FPS", description="First Person Shooter")
    vgt_repo.add(vgt)
    uow.commit()
    
    # Crear Torneo (Reading all records de catálogos internamente)
    nuevo_torneo = Tournament(
        name="Blast Premier 2026",
        start_date=date(2026, 6, 1),
        id_videogame_type=vgt.id_videogame_type,
        id_region=1,
        id_tournament_mode=1,
        id_tournament_status=1
    )
    tour_repo.add(nuevo_torneo)
    uow.commit()
    
    # Specific Query: Get top players (aunque esté vacía la tabla de matches)
    top = tour_repo.get_top_players(nuevo_torneo.id_tournament, limit=3)
    print(f"[OK] Specific Query (Top Players): Ejecutada correctamente")

In [ ]:
print("--- VERIFYING ALEMBIC MIGRATION (last_update) ---")
with get_uow() as uow:
    reg = uow.session.query(Region).first()
    if hasattr(reg, 'last_update'):
        print(f"✅ Campo 'last_update' detectado: {reg.last_update}")
    else:
        print("❌ Error: No se detecta el campo last_update")